# TaxGPT — Episode 3: Token & Positional Embeddings

Companion notebook to blog post *"Token Embeddings and Positional Embeddings Explained (TaxGPT Episode 3)"*.

> **Same caveat as the blog post:** the dimensions used here (768-dim embeddings, 1024-token context, 50,257 vocab) are TaxGPT's current architecture spec. Cross-check against the live model code before treating these as final.

Reference: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Ch. 2. Andrej Karpathy, *Building makemore Part 2: MLP*.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

VOCAB_SIZE = 50_257   # GPT-2 BPE vocab (Episode 2)
EMB_DIM    = 768
CONTEXT_LEN = 1024

token_embedding = nn.Embedding(VOCAB_SIZE, EMB_DIM)
positional_embedding = nn.Embedding(CONTEXT_LEN, EMB_DIM)

print("token_embedding weight shape:", token_embedding.weight.shape)
print("positional_embedding weight shape:", positional_embedding.weight.shape)

token_embedding weight shape: torch.Size([50257, 768])
positional_embedding weight shape: torch.Size([1024, 768])


## 1. From token IDs to vectors

Take a short sequence of token IDs (stand-ins for real BPE output from Episode 2) and look up their embeddings.

In [2]:
# pretend these are BPE token ids for "input tax credit is available"
token_ids = torch.tensor([[818, 1687, 3884, 318, 1695]])  # shape: (batch=1, seq_len=5)
batch_size, seq_len = token_ids.shape

tok_emb = token_embedding(token_ids)
print("input token_ids shape:", token_ids.shape)
print("tok_emb shape:", tok_emb.shape, " -> (batch, seq_len, emb_dim)")

input token_ids shape: torch.Size([1, 5])
tok_emb shape: torch.Size([1, 5, 768])  -> (batch, seq_len, emb_dim)


## 2. Positional embeddings — same lookup, but indexed by position, not identity

In [3]:
positions = torch.arange(seq_len)
pos_emb = positional_embedding(positions)  # shape: (seq_len, emb_dim)

print("positions:", positions)
print("pos_emb shape:", pos_emb.shape)

# broadcasts across the batch dimension when added to tok_emb
input_embedding = tok_emb + pos_emb
print("input_embedding shape:", input_embedding.shape, " -> identity + position, fused")

positions: tensor([0, 1, 2, 3, 4])
pos_emb shape: torch.Size([5, 768])
input_embedding shape: torch.Size([1, 5, 768])  -> identity + position, fused


## 3. Proving order actually matters now

Same two token IDs, reversed order. Token embeddings alone are identical (just permuted); once positions are added, the two sequences produce genuinely different vectors at each slot.

In [4]:
seq_a = torch.tensor([[100, 200, 300]])
seq_b = torch.tensor([[300, 200, 100]])  # reversed

emb_a = token_embedding(seq_a) + positional_embedding(torch.arange(3))
emb_b = token_embedding(seq_b) + positional_embedding(torch.arange(3))

# token embedding alone: same set of vectors, just permuted
tok_only_a = token_embedding(seq_a).squeeze(0)
tok_only_b = token_embedding(seq_b).squeeze(0)
print("token-embedding-only cosine sim, position 0 vs 0 (a vs b):",
      torch.cosine_similarity(tok_only_a[0], tok_only_b[0], dim=0).item())

# with positional info fused in, position-0 vectors diverge because the underlying tokens differ
print("full input-embedding cosine sim, position 0 vs 0 (a vs b):",
      torch.cosine_similarity(emb_a[0,0], emb_b[0,0], dim=0).item())

token-embedding-only cosine sim, position 0 vs 0 (a vs b): 0.0016047479584813118
full input-embedding cosine sim, position 0 vs 0 (a vs b): 0.5268091559410095


## 4. Where the parameters actually go

The blog post's headline number: the token embedding table alone is ~38.6M parameters — a meaningful chunk of TaxGPT's 131M total.

In [5]:
tok_emb_params = VOCAB_SIZE * EMB_DIM
pos_emb_params = CONTEXT_LEN * EMB_DIM
total_taxgpt_params = 131_497_728  # real, verified param count from the training run

print(f"token embedding table:     {tok_emb_params:,} params")
print(f"positional embedding table: {pos_emb_params:,} params")
print(f"combined embedding params:  {tok_emb_params + pos_emb_params:,} params")
print(f"share of full 131M model:   {(tok_emb_params + pos_emb_params) / total_taxgpt_params:.1%}")

token embedding table:     38,597,376 params
positional embedding table: 786,432 params
combined embedding params:  39,383,808 params
share of full 131M model:   30.0%


## Takeaway

Embeddings give each token a starting representation *in isolation*. Tokens still can't "see" each other yet — that interaction is the job of self-attention.

**Next notebook: Episode 4 — Self-attention, built from the actual matrix math.**